### Gen answers

In [ ]:
import os
import json
import base64
import time
from dotenv import load_dotenv
from tqdm import tqdm
from mistralai.client import Mistral # Dùng SDK mới nhất của Mistral
from PIL import Image
import io
import base64

# Nạp các biến từ file .env
load_dotenv()
MISTRAL_API_KEY = "YOUR_API_KEY"
MODEL_ID = "pixtral-12b-2409"

# Khởi tạo client Mistral
client = Mistral(api_key=MISTRAL_API_KEY)

def resize_and_encode_image(image_path, max_size=512):
    """
    Đọc ảnh, thu nhỏ nếu kích thước lớn hơn max_size và chuyển sang Base64
    """
    try:
        with Image.open(image_path) as img:
            # Chuyển đổi sang RGB để tránh lỗi với ảnh PNG có nền trong suốt
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # thumbnail() tự động giữ nguyên tỷ lệ khung hình (aspect ratio)
            img.thumbnail((max_size, max_size))
            
            # Lưu vào bộ nhớ đệm (buffer) thay vì ghi đè ra ổ cứng
            buffered = io.BytesIO()
            # quality=85 giúp nén dung lượng mà vẫn giữ độ nét tốt
            img.save(buffered, format="JPEG", quality=85) 
            
            # Chuyển đổi sang chuỗi Base64
            img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
            return img_str
    except Exception as e:
        print(f"Lỗi khi xử lý ảnh {image_path}: {e}")
        return None
def get_mistral_answer(image_path, question):
    """Gọi API bằng thư viện Mistral chính thức"""
    system_prompt = (
  """Bạn là một trợ lý AI xuất sắc trong việc phân tích hình ảnh (VQA).
Nhiệm vụ của bạn là trả lời câu hỏi của người dùng dựa trên hình ảnh được cung cấp một cách ngắn gọn, chính xác bằng Tiếng Việt một cách tự nhiên.

QUY TẮC QUAN TRỌNG:
1. Bạn phải CỐ GẮNG HẾT SỨC để trả lời dựa trên bất kỳ chi tiết nào bạn có thể nhận diện được trong ảnh,một cách ngắn gọn , xúc tích.
2. CHỈ KHI hình ảnh THỰC SỰ QUÁ MỜ, nhòe, nhiễu nặng hoặc tối đen đến mức MỘT CON NGƯỜI cũng không thể đoán được bất cứ thông tin gì liên quan đến câu hỏi,thì bạn phải trả lời đúng một câu: "Ảnh mờ, bạn vui lòng chụp lại ảnh rõ hơn."
3. Nếu ảnh chỉ hơi mờ hoặc thiếu sáng nhưng vẫn có thể suy luận được, tuyệt đối không phàn nàn, hãy đưa ra câu trả lời tốt nhất của bạn.
4. Trả lời ngắn gọn, súc tích trong một câu, không giải thích dài dòng, không thêm thông tin không cần thiết, chỉ trả lời đúng trọng tâm câu hỏi.
5. Luôn trả lời bằng tiếng Việt, không được trả lời bằng ngôn ngữ khác, không được trả lời bằng tiếng Anh, không được trả lời bằng tiếng Trung, chỉ được trả lời bằng tiếng Việt.
"""    )
    
    try:
        base64_image = resize_and_encode_image(image_path)
        
        # Cấu trúc messages chuẩn theo tài liệu Mistral
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", 
                        "text": f"{system_prompt}\n\nQuestion: {question}"
                    },
                    {
                        "type": "image_url", 
                        "image_url": f"data:image/jpeg;base64,{base64_image}"
                    }
                ]
            }
        ]
        
        # Gọi SDK
        response = client.chat.complete(
            model=MODEL_ID,
            messages=messages,
            temperature=0.1,
            max_tokens=100
        )
        
        content = response.choices[0].message.content
        if not content:
            return "Lỗi: API Mistral trả về kết quả rỗng."
            
        return content

    except Exception as e:
        return f"Lỗi xử lý API: {str(e)}"

def process_dataset(json_path, output_filename, save_dir):
    """Đọc file JSON, gọi API, lưu file ra thư mục chỉ định và nghỉ 2s tránh Rate Limit"""
    base_dir = os.path.dirname(json_path)
    
    # Tạo thư mục lưu nếu chưa tồn tại
    os.makedirs(save_dir, exist_ok=True)
    output_path = os.path.join(save_dir, output_filename)
    
    # CƠ CHẾ RESUME
    if os.path.exists(output_path):
        print(f"\n[INFO] Tìm thấy file đang lưu dở: {output_path}. Sẽ chạy tiếp tục (Resume).")
        with open(output_path, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    else:
        print(f"\n[INFO] Đang bắt đầu xử lý tập dữ liệu mới: {json_path}")
        with open(json_path, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
            
    print(f"Tổng số mẫu: {len(dataset)}")
    
    for item in tqdm(dataset, desc="Đang sinh câu trả lời"):
        image_name = item.get("image")
        question = item.get("question")
        
        # Bỏ qua nếu đã có đáp án xịn
        existing_ans = item.get("AI_answer")
        if existing_ans and not str(existing_ans).startswith("Lỗi"):
            continue
            
        # Vẫn tìm ảnh ở thư mục base_dir (nơi chứa file json gốc)
        image_path = os.path.join(base_dir, image_name)
        
        if os.path.exists(image_path):
            ai_answer = get_mistral_answer(image_path, question)
            # Thêm delay 2 giây ngay sau khi gửi request thành công (tránh bị hết rpm)
            time.sleep(2)
        else:
            ai_answer = "Lỗi: Không tìm thấy file ảnh gốc trên ổ cứng."
            
        item["AI_answer"] = str(ai_answer)
        
        # In log
        short_ans = item["AI_answer"][:60] + "..." if len(item["AI_answer"]) > 60 else item["AI_answer"]
        tqdm.write(f"[LOG] {image_name} | Q: {question} -> A: {short_ans}")
        
        # Lưu file ngay lập tức vào thư mục
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, ensure_ascii=False, indent=4)
            
    print(f"Đã hoàn thành toàn bộ và lưu kết quả tại: {output_path}")

# THỰC THI CHƯƠNG TRÌNH
if __name__ == "__main__":
    # NGUỒN VÀO (Ảnh và metadata)
    file_ans_0 = "vizwiz_dataset_1000/answerable_0/metadata_ans_0.json"
    file_ans_1 = "vizwiz_dataset_1000/answerable_1/metadata_ans_1.json"
    
    # NGUỒN RA (Lưu file json mới)
    save_directory = "vizwiz_dataset_1000"
    
    if os.path.exists(file_ans_0):
        process_dataset(file_ans_0, "pixtral_ans_0.json", save_directory)
    else:
        print(f"Không tìm thấy file nguồn: {file_ans_0}")
        
    if os.path.exists(file_ans_1):
        process_dataset(file_ans_1, "pixtral_ans_1.json", save_directory)
    else:
        print(f"Không tìm thấy file nguồn: {file_ans_1}")


[INFO] Đang bắt đầu xử lý tập dữ liệu mới: vizwiz_dataset_1000/answerable_0/metadata_ans_0.json
Tổng số mẫu: 400


Đang sinh câu trả lời:   0%|          | 1/400 [00:03<20:32,  3.09s/it]

[LOG] IMG-0-0000001.jpg | Q: What is this device? -> A: Ảnh mờ, nhưng có vẻ như là một loa hình tròn hoặc thiết bị â...


Đang sinh câu trả lời:   0%|          | 2/400 [00:06<21:13,  3.20s/it]

[LOG] IMG-0-0000002.jpg | Q: What's in this box? -> A: Hình ảnh không rõ nét về phần hộp, nhưng phần nhìn thấy được...


...

[LOG] IMG-1-0000598.jpg | Q: who is this? -> A: Đây là nhân vật Shepard từ trò chơi "Mass Effect 3" với tran...


Đang sinh câu trả lời: 100%|█████████▉| 599/600 [33:54<00:03,  3.51s/it]

[LOG] IMG-1-0000599.jpg | Q: What does the corridor look like? -> A: Đây là một hành lang rộng rãi, sạch sẽ, sàn nhẵn màu trắng v...


Đang sinh câu trả lời: 100%|██████████| 600/600 [33:57<00:00,  3.40s/it]

[LOG] IMG-1-0000600.jpg | Q: What is this? -> A: Hình ảnh này là gói súp gà Campbell's Real Stock.
Đã hoàn thành toàn bộ và lưu kết quả tại: vizwiz_dataset_1000/mistral_ans_1.json
